/kaggle/input/datasets/sabinasabitovaseds/100wlasl/WLASL_100/

In [23]:
import os
import cv2
import time
import torch
import torch.nn as nn
import numpy as np
from torch.utils.data import Dataset, DataLoader

DATA_DIR = "/kaggle/input/datasets/sabinasabitovaseds/100wlasl/WLASL_100/"
BATCH_SIZE = 8 
LR = 1e-4      
EPOCHS = 20
SAVE_PATH = "best_slowfast_wlasl100.pth"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class WLASLDataset(Dataset):
    def __init__(self, root_dir, mode='train'):
        self.root_dir = os.path.join(root_dir, mode)
        self.mode = mode
        if not os.path.exists(self.root_dir):
            raise FileNotFoundError(f"Path not found: {self.root_dir}")
            
        self.classes = sorted(os.listdir(self.root_dir))
        self.class_to_idx = {cls: i for i, cls in enumerate(self.classes)}
        self.samples = []
        for cls in self.classes:
            cls_path = os.path.join(self.root_dir, cls)
            for vid in os.listdir(cls_path):
                if vid.endswith('.mp4'):
                    self.samples.append((os.path.join(cls_path, vid), self.class_to_idx[cls]))

    def __len__(self):
        return len(self.samples)

    def _get_frames(self, video_path):
        cap = cv2.VideoCapture(video_path)
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        window_size = 64
        
        if self.mode == 'train':
            start_frame = np.random.randint(0, max(1, total_frames - window_size))
        else:
            start_frame = max(0, (total_frames - window_size) // 2)

        buffer = []
        cap.set(cv2.CAP_PROP_POS_FRAMES, start_frame)
        for _ in range(window_size):
            ret, frame = cap.read()
            if not ret:
                frame = buffer[-1] if len(buffer) > 0 else np.zeros((224, 224, 3), np.uint8)
            else:
                frame = cv2.resize(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB), (224, 224))
            buffer.append(frame)
        cap.release()
        return np.array(buffer)

    def __getitem__(self, idx):
        video_path, label = self.samples[idx]
        raw_clip = self._get_frames(video_path) # This returns 64 frames
        
        raw_clip = raw_clip.astype(np.float32) / 255.0
        raw_clip = (raw_clip - 0.45) / 0.225 

        slow_stride = 8
        slow_idx = np.arange(8) * slow_stride
        slow = torch.from_numpy(raw_clip[slow_idx]).permute(3, 0, 1, 2)
        
        fast_stride = 2 
        fast_idx = np.arange(32) * fast_stride
        fast = torch.from_numpy(raw_clip[fast_idx]).permute(3, 0, 1, 2)
        
        return [slow, fast], label

def build_slowfast_wlasl(num_classes=100):
    model = torch.hub.load('facebookresearch/pytorchvideo', 'slowfast_r50', pretrained=True)
    in_features = model.blocks[6].proj.in_features
    model.blocks[6].proj = nn.Linear(in_features, num_classes)
    return model

train_loader = DataLoader(WLASLDataset(DATA_DIR, 'train'), batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(WLASLDataset(DATA_DIR, 'val'), batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader = DataLoader(WLASLDataset(DATA_DIR, 'test'), batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

model = build_slowfast_wlasl(num_classes=100).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
best_val_acc = 0.0

print(f"{'Epoch':<8} {'Train Loss':<12} {'Train Acc':<12} {'Val Loss':<12} {'Val Acc':<12}")
print("-" * 70)

for epoch in range(EPOCHS):
    epoch_start_time = time.time()
    
    model.train()
    train_loss, train_correct, train_total = 0, 0, 0
    for inputs, labels in train_loader:
        inputs = [i.to(device) for i in inputs]
        labels = labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
        train_correct += (torch.max(outputs, 1)[1] == labels).sum().item()
        train_total += labels.size(0)

    model.eval()
    val_loss, val_correct, val_total = 0, 0, 0
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs = [i.to(device) for i in inputs]
            labels = labels.to(device)
            outputs = model(inputs)
            val_loss += criterion(outputs, labels).item()
            val_correct += (torch.max(outputs, 1)[1] == labels).sum().item()
            val_total += labels.size(0)

    avg_val_acc = 100 * val_correct / val_total
    save_msg = ""
    if avg_val_acc > best_val_acc:
        best_val_acc = avg_val_acc
        torch.save(model.state_dict(), SAVE_PATH)
        save_msg = "*"

    print(f"{epoch+1:<8} {train_loss/len(train_loader):<12.4f} {100*train_correct/train_total:<12.2f} "
          f"{val_loss/len(val_loader):<12.4f} {avg_val_acc:<12.2f} {save_msg}")

print("-" * 70)
print("Training Complete. Running Final Evaluation on TEST set...")
model.load_state_dict(torch.load(SAVE_PATH)) 
model.eval()
test_correct, test_total = 0, 0
with torch.no_grad():
    for inputs, labels in test_loader:
        inputs = [i.to(device) for i in inputs]
        labels = labels.to(device)
        outputs = model(inputs)
        test_correct += (torch.max(outputs, 1)[1] == labels).sum().item()
        test_total += labels.size(0)

print(f"Final Test Accuracy: {100 * test_correct / test_total:.2f}%")

Using cache found in /root/.cache/torch/hub/facebookresearch_pytorchvideo_main


Epoch    Train Loss   Train Acc    Val Loss     Val Acc     
----------------------------------------------------------------------
1        4.5790       2.70         4.1345       9.47         *
2        3.9577       11.58        3.2670       25.15        *
3        3.0325       27.60        2.4949       33.14        *
4        2.1446       48.68        2.0720       44.08        *
5        1.4045       68.52        1.7147       52.07        *
6        0.8801       81.83        1.6992       56.21        *
7        0.5132       91.47        1.6376       55.92        
8        0.3429       94.59        1.6124       60.65        *
9        0.2054       97.64        1.6538       57.99        
10       0.1452       98.27        1.5335       58.88        
11       0.0941       98.96        1.4658       63.31        *
12       0.0744       99.17        1.4306       65.09        *
13       0.0634       99.17        1.4663       64.50        
14       0.0783       98.68        1.6061       61.83

In [15]:
!pip install fvcore av pytorchvideo

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 2.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.7/132.7 kB 5.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 3.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.4/36.4 MB 58.4 MB/s eta 0:00:00:00:0100:01
  Created wheel for fvcore: filename=fvcore-0.1.5.post20221221-py3-none-any.whl size=61397 sha256=9be120204342eca96fb2ec021344457631cd53b043c3b3dfa3bf60f4f09951c0
  Stored in directory: /root/.cache/pip/wheels/ed/9f/a5/e4f5b27454ccd4596bd8b62432c7d6b1ca9fa22aef9d70a16a
  Created wheel for pytorchvideo: filename=pytorchvideo-0.1.5-py3-none-any.whl size=188686 sha256=0466a73b27ecd70531e91b5d4fea2c585c4b0b829cfbb94a8b9882a4fa65ea8b
  Stored in directory: /root/.cache/pip/wheels/b3/49/dc/aab2dce83e38b59849db13a4f4ddd220e568e24b58332fb0f9
  Crea